# 1. Setup

The single-head counterpart to frozen 3B. Keep the same two frozen DINOv2 towers, frozen CLIP text encoder, prompts, shared split, seed and epoch budget. Train only the joint image-to-pair text projection and its temperature.

In [1]:
import gc, json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3b_dino_text_alignment_single_head', 'B'
DEBUG = False
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          pair_lr=1e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256,
          text_batch_size=16, dino_revision='main', text_revision='main',
          pair_prompt='a photo of a {crop} leaf with {disease}',
          healthy_prompt='a photo of a healthy {crop} leaf')

EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 4  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

# 2. Data
Direct common loaders: formal7223 train/777 validation,323 development pairs. DEBUG uses common subsets. No extra holdout, sampler or augmentation. No official test CSV is read here.


In [2]:
# Same common data path as the old CLIP notebook; DINO uses imagenet normalization.
frames = [load_split(cfg.TRAIN_CSV)]
assert len(frames[0]) == 8000
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_SEEN_PAIRS, 2)
concept_names = get_crop_and_disease_names()
def loaders():
    cfg.set_seed(cfg.SEED)
    return get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
train_loader, val_loader, seen_loader, unseen_loader = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(train=len(train_loader.dataset), val=len(val_loader.dataset))
print('Device:', cfg.DEVICE, 'Data:', cfg.DATA_DIR, 'Samples:', sample_counts)


Device: cuda Data: C:\Users\TABLE #012\Desktop\COS30082-Plant-Disease-OOD\data Samples: {'train': 7223, 'val': 777}


# 3. Model

Average the two frozen visual feature vectors, then project into the frozen CLIP text space. One head scores each candidate crop–disease prompt by scaled cosine similarity. Development encodes only the 323 training prompts.

In [3]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    @torch.no_grad()
    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

class TextOnly:
    """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
    def __init__(self,config,device,allow_download=False,revision=None):
        from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
        path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
            ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
        self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
        self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
        if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
        self.dimension=self.model.config.projection_dim;self.device=device
        if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
        weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,
            class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
    @torch.no_grad()
    def encode(self,prompts,size):
        self.assert_frozen();result=[]
        for start in range(0,len(prompts),size):
            tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
            result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
        return torch.cat(result)
    def assert_frozen(self):
        if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
            raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')


def pair_text_bank(text, candidates, names):
    crops, diseases = names
    prompts = []
    for crop_id, disease_id in candidates.tolist():
        crop = crops[crop_id].replace('_', ' ')
        disease = diseases[disease_id].replace('_', ' ')
        template = HP['healthy_prompt'] if disease.strip().casefold() == 'healthy' else HP['pair_prompt']
        prompts.append(template.format(crop=crop, disease=disease))
    return text.encode(prompts, HP['text_batch_size'])


RECIPE = dict(pair=1.)

class Model(nn.Module):
    def __init__(self, backbone, bank):
        super().__init__()
        self.backbone = backbone
        self.pair_projection = nn.Sequential(
            nn.Linear(backbone.width, HP['hidden_dim']), nn.ReLU(),
            nn.Linear(HP['hidden_dim'], bank.shape[1]))
        self.log_temperature = nn.Parameter(torch.tensor(math.log(1 / .07)))
        self.register_buffer('pair_text', bank)

    def forward(self, pixels):
        crop, disease = self.backbone(pixels)
        image = F.normalize(self.pair_projection((crop + disease) / 2), dim=1)
        return self.log_temperature.clamp(0., math.log(100.)).exp() * (image @ self.pair_text.T)


def make_model(candidates, names, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.requires_grad_(False).eval()
    text = TextOnly(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('text', {}).get('revision'))
    bank = pair_text_bank(text, candidates, names)
    cfg.set_seed(cfg.SEED)
    model = Model(backbone, bank).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, text, dict(dino=backbone.metadata, text=text.metadata)


def make_predict(model):
    @torch.no_grad()
    def predict(images):
        model.eval()
        return model(images.to(cfg.DEVICE))
    return predict

# 4. Training
Train one complete approach using the baseline training budget and common validation for checkpoint selection. No official test data is accessed.


In [4]:
OUT.mkdir(parents=True, exist_ok=False)
cfg.set_seed(cfg.SEED)
train_loader, val_loader, _, _ = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
model, text, metadata = make_model(pairs, concept_names)
recipe = RECIPE
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=HP['pair_lr'], weight_decay=HP['weight_decay'])
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
history = []; best_val = -1.; checkpoint_path = OUT / (MODEL_NAME + '.pt')
for epoch in range(EPOCHS):
    model.train(); model.backbone.eval()
    total = 0.; samples = 0
    for images, _, _, labels in train_loader:
        optimizer.zero_grad(set_to_none=True)
        scores = model(images.to(cfg.DEVICE))
        value = criterion(scores, labels.to(cfg.DEVICE))
        if not torch.isfinite(value): raise RuntimeError('Nonfinite loss.')
        value.backward()
        if text is not None: text.assert_frozen()
        if any(p.grad is not None for p in model.backbone.parameters() if not p.requires_grad):
            raise RuntimeError('Frozen DINO received gradients.')
        gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not gradients or not all(torch.isfinite(g).all() for g in gradients):
            raise RuntimeError('Invalid training gradients.')
        optimizer.step()
        total += value.item() * len(labels); samples += len(labels)
    schedule.step()
    predict = make_predict(model)
    validation = val_accuracy(predict, val_loader)
    history.append(dict(epoch=epoch+1, loss=total/samples, val_top1=validation))
    print(history[-1])
    if validation > best_val:
        best_val = validation
        torch.save(dict(model_name=MODEL_NAME,
            state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
            hp=HP, debug=DEBUG, family=FAMILY, recipe=recipe,
            pairs=pairs, metadata=metadata, best_val=best_val,
            best_epoch=epoch+1, environment=environment), checkpoint_path)
del predict, model, text, optimizer, schedule
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
selection = dict(model_name=MODEL_NAME, family=FAMILY, hp=HP, debug=DEBUG,
    recipe=RECIPE, checkpoint=checkpoint_path.name, best_val=best_val, history=history,
    train_images=train_loader.dataset.frame.image.tolist(),
    validation_images=val_loader.dataset.frame.image.tolist(),
    development_pairs=pairs.tolist(), criterion='common validation Top1; earliest epoch on ties')
with (OUT/'selection.json').open('x', encoding='utf8') as file: json.dump(selection, file, indent=2)
print('Selection frozen:', OUT/'selection.json')

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

C:\Users\TABLE #012\Desktop\COS30082-Plant-Disease-OOD\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TABLE #012\.cache\huggingface\hub\models--timm--vit_base_patch14_dinov2.lvd142m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/346M [00:00<?, ?B/s]

... (8 outputs left out)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

{'epoch': 1, 'loss': 3.7130922519911462, 'val_top1': 51.87}
{'epoch': 2, 'loss': 2.8473249753888386, 'val_top1': 59.2}
{'epoch': 3, 'loss': 2.517869219165355, 'val_top1': 62.29}
{'epoch': 4, 'loss': 2.309991472004755, 'val_top1': 67.7}
{'epoch': 5, 'loss': 2.147369555781361, 'val_top1': 71.69}
{'epoch': 6, 'loss': 2.023577768387069, 'val_top1': 73.62}
{'epoch': 7, 'loss': 1.9211776318616787, 'val_top1': 75.55}
{'epoch': 8, 'loss': 1.8290978139512644, 'val_top1': 75.16}
{'epoch': 9, 'loss': 1.759706046035642, 'val_top1': 78.12}
{'epoch': 10, 'loss': 1.7020431489936536, 'val_top1': 78.25}
{'epoch': 11, 'loss': 1.6515544753614, 'val_top1': 78.76}
{'epoch': 12, 'loss': 1.6116871203673806, 'val_top1': 78.51}
{'epoch': 13, 'loss': 1.5737021809120713, 'val_top1': 79.02}
{'epoch': 14, 'loss': 1.5482663322845094, 'val_top1': 79.79}
{'epoch': 15, 'loss': 1.527608483778082, 'val_top1': 80.05}
{'epoch': 16, 'loss': 1.5087598589307365, 'val_top1': 80.31}
{'epoch': 17, 'loss': 1.4999599264679102, 'v

# 5. Evaluation
Same common.evaluate as the old CLIP notebook. After ALL choices are frozen, enable RUN_FINAL_EVALUATION. Official333 candidates appear only here. The two extra component accuracies use the SAME clean unseen predictions, not robustness outputs; they describe unseen-combination test images, not genuinely unseen crop/disease classes.


In [7]:
RUN_FINAL_EVALUATION = True  # Enable only after all model choices are frozen.
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    run_folder = OUT
    selection = json.loads((run_folder / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run_folder / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=MODEL_NAME, family=FAMILY, debug=False, hp=HP, recipe=RECIPE)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError('The selected run or checkpoint does not match the frozen settings.')
    if selection['development_pairs'] != pairs.tolist() or not torch.equal(checkpoint['pairs'], pairs):
        raise ValueError('Development candidates changed.')
    if (selection['train_images'] != train_loader.dataset.frame.image.tolist()
            or selection['validation_images'] != val_loader.dataset.frame.image.tolist()):
        raise ValueError('Common train/validation membership changed.')
    model, text, metadata = make_model(pairs, concept_names, checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    # Only now open the official candidates and test loaders.
    final_pairs = get_pairs(final_evaluation=True)
    model.pair_text = pair_text_bank(text, final_pairs, concept_names)
    seen_loader, unseen_loader = get_final_test_loaders(
        HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    sample_counts.update(seen=len(seen_loader.dataset), unseen=len(unseen_loader.dataset))
    predict_final = make_predict(model)
    # common.evaluate predicts clean seen, then clean unseen, then robustness.
    seen_count = sample_counts['seen']
    clean_end = seen_count + sample_counts['unseen']
    unseen_batches, processed = [], 0
    def predict(images):
        global processed
        scores = predict_final(images)
        if seen_count <= processed < clean_end:
            unseen_batches.append(scores.float().cpu())
        processed += len(images)
        return scores
    RESULTS = cfg.RESULTS_DIR / run_folder.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust=True)
    finally:
        cfg.RESULTS_DIR = previous_results
    unseen_labels = torch.as_tensor(unseen_loader.dataset.frame.pair_row.to_numpy())
    crop_acc, disease_acc = concept_accuracy(torch.cat(unseen_batches), unseen_labels, final_evaluation=True)
    result.update(unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc,
                  debug=DEBUG, best_val=checkpoint['best_val'])
    reports.append(result)
    del predict_final, model, text, unseen_batches

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "model": "approach3b_dino_text_alignment_single_head",
  "seen_top1": 65.85,
  "seen_top5": 87.5,
  "unseen_top1": 6.7,
  "unseen_top5": 59.53,
  "harmonic_mean": 12.16,
  "crop_acc": 93.0,
  "disease_acc": 67.25,
  "robustness": {
    "brightness": {
      "top1": 65.4,
      "drop": 0.45
    },
    "blur": {
      "top1": 62.7,
      "drop": 3.15
    },
    "noise": {
      "top1": 58.55,
      "drop": 7.3
    }
  }
}


# 6. Save
Same summary fields as the old CLIP notebook. Separate run folders prevent overwriting baselines. Keep checkpoints and result JSON/CSV/confusion files. One common seed does not establish statistical significance; formal results are not generated during authoring.


In [8]:
if RUN_FINAL_EVALUATION:
    for result in reports:
        (RESULTS/(result['model']+'.json')).write_text(json.dumps(result, indent=2), encoding='utf8')
    summary = dict(selection=selection, results=reports, hp=HP,
                   environment=environment, samples=sample_counts)
    (RESULTS/(MODEL_NAME+'_comparison.json')).write_text(json.dumps(summary, indent=2), encoding='utf8')
    table = pd.DataFrame(reports)
    table.to_csv(RESULTS/(MODEL_NAME+'_comparison.csv'), index=False)
    display(table[['model','seen_top1','unseen_top1','harmonic_mean',
                   'unseen_test_crop_acc','unseen_test_disease_acc']])
    print('Checkpoints:', run_folder, 'Results:', RESULTS)
else:
    print('Save the complete checkpoint run folder after training:', OUT)


,model,seen_top1,unseen_top1,harmonic_mean,unseen_test_crop_acc,unseen_test_disease_acc
0,approach3b_dino_text_alignment_single_head,65.85,6.7,12.16,79.9,8.7


Checkpoints: C:\Users\TABLE #012\Desktop\COS30082-Plant-Disease-OOD\checkpoints\approach3b_dino_text_alignment_single_head_20261126_152217_756440 Results: C:\Users\TABLE #012\Desktop\COS30082-Plant-Disease-OOD\results\approach3b_dino_text_alignment_single_head_20261126_152217_756440
